# Experiment 2: Data Preprocessing, Pipelines and Leakage Detection

**Dataset:** Titanic passengers  
**Target:** `Survived`  
**Experiment type:** Diagnostic comparison

## Goal

Identify preprocessing leakage, reproduce flawed workflows, rebuild the workflow correctly, and compare held-out test metrics.

## 1. Predict — likely leakage points

Leakage is likely wherever `fit`, `fit_transform`, or a data-derived statistic is computed **before** the train/test split. A scaler, imputer, or encoder fitted on all rows exposes the training process to the test distribution. Feature selection, resampling, and target-derived features can violate the boundary too.

**General evaluation rule:** split raw data first; fit all stateful preprocessing and the estimator using only training rows; use the test set only for final evaluation.

## 2. Setup and inspect inputs

In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
TEST_SIZE = 0.30

DATA_PATH = Path("Titanic-Dataset.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("Week 3") / "Titanic-Dataset.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError("Titanic-Dataset.csv was not found in the current folder or Week 3.")

df = pd.read_csv(DATA_PATH)
print(f"Source: {DATA_PATH.resolve()}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
display(df.head())

Source: D:\Programming\SEM-5 ML labs\Week 3\Titanic-Dataset.csv
Shape: 891 rows x 12 columns


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
numeric_columns = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_columns = ["Sex", "Embarked"]
feature_columns = numeric_columns + categorical_columns
X = df[feature_columns].copy()
y = df["Survived"].copy()
display(X.isna().sum().rename("missing_count").to_frame())
print("Target distribution:")
display(y.value_counts(normalize=True).sort_index().to_frame("proportion"))

,missing_count
Pclass,0
Age,177
SibSp,0
Parch,0
Fare,0
Sex,0
Embarked,2


Target distribution:


,proportion
Survived,
0,0.616162
1,0.383838


## 3. Inspect and run the flawed pipeline

The classic bug is `scaler.fit_transform(X)` before `train_test_split`. Here the deliberately flawed version fits **imputation, scaling, and encoding** on the full feature matrix before splitting. Thus full-data medians, modes, means, standard deviations, and category vocabulary influence the representation used for training.

In [3]:
def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_columns),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_columns),
    ])

# INTENTIONALLY FLAWED: learns preprocessing statistics from all rows.
leaky_preprocessor = make_preprocessor()
X_leaky = leaky_preprocessor.fit_transform(X)
X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
leaky_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
leaky_model.fit(X_train_leaky, y_train_leaky)
leaky_accuracy = accuracy_score(y_test_leaky, leaky_model.predict(X_test_leaky))
print(f"Leaky full-preprocessing test accuracy: {leaky_accuracy:.3f}")

Leaky full-preprocessing test accuracy: 0.799


## 4. Rebuild correctly — split first

The raw data is split first. `Pipeline.fit(X_train, y_train)` then learns all imputation values, scaling parameters, category levels, and model coefficients strictly from training rows.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
corrected_pipeline = Pipeline([
    ("preprocessor", make_preprocessor()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
corrected_pipeline.fit(X_train, y_train)
corrected_accuracy = accuracy_score(y_test, corrected_pipeline.predict(X_test))
print(f"Leaky pipeline test accuracy:     {leaky_accuracy:.3f}")
print(f"Corrected pipeline test accuracy: {corrected_accuracy:.3f}")
print(f"Difference (leaky - corrected):   {leaky_accuracy - corrected_accuracy:+.3f}")

Leaky pipeline test accuracy:     0.799
Corrected pipeline test accuracy: 0.799
Difference (leaky - corrected):   +0.000


In [ ]:
# Inspect training-only imputation and verify that both transformed partitions are complete.
correct_num_imputer = corrected_pipeline.named_steps["preprocessor"].named_transformers_["num"].named_steps["imputer"]
correct_cat_imputer = corrected_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["imputer"]

imputation_values = pd.DataFrame({
    "feature": numeric_columns + categorical_columns,
    "strategy": ["median"] * len(numeric_columns) + ["most_frequent"] * len(categorical_columns),
    "training_value": list(correct_num_imputer.statistics_) + list(correct_cat_imputer.statistics_),
})
display(imputation_values)

train_processed = corrected_pipeline.named_steps["preprocessor"].transform(X_train)
test_processed = corrected_pipeline.named_steps["preprocessor"].transform(X_test)
assert pd.isna(train_processed.toarray() if hasattr(train_processed, "toarray") else train_processed).sum() == 0
assert pd.isna(test_processed.toarray() if hasattr(test_processed, "toarray") else test_processed).sum() == 0
print("Missing values after corrected imputation: train=0, test=0")

## 5. Modify — second leakage source: full-data imputation

This flawed variant computes median/mode values from the full dataset before splitting, then fits scaling, encoding, and the model only on training data. It isolates full-data imputation as another leakage point.

In [5]:
# INTENTIONALLY FLAWED: held-out rows influence these fill values.
X_imputed_full = X.copy()
X_imputed_full["Age"] = X_imputed_full["Age"].fillna(X_imputed_full["Age"].median())
X_imputed_full["Fare"] = X_imputed_full["Fare"].fillna(X_imputed_full["Fare"].median())
X_imputed_full["Embarked"] = X_imputed_full["Embarked"].fillna(X_imputed_full["Embarked"].mode().iloc[0])

X_train_imp, X_test_imp, y_train_imp, y_test_imp = train_test_split(
    X_imputed_full, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
after_imputation = ColumnTransformer([
    ("num", StandardScaler(), numeric_columns),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
])
imputation_leaky_pipeline = Pipeline([
    ("preprocessor", after_imputation),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
imputation_leaky_pipeline.fit(X_train_imp, y_train_imp)
imputation_leaky_accuracy = accuracy_score(y_test_imp, imputation_leaky_pipeline.predict(X_test_imp))

comparison = pd.DataFrame({
    "workflow": [
        "All preprocessing before split (leaky)",
        "Only imputation before split (leaky)",
        "Training-only pipeline (correct)",
    ],
    "test_accuracy": [leaky_accuracy, imputation_leaky_accuracy, corrected_accuracy],
    "trustworthy": [False, False, True],
})
display(comparison.style.format({"test_accuracy": "{:.3f}"}))

,workflow,test_accuracy,trustworthy
0,All preprocessing before split (leaky),0.799,False
1,Only imputation before split (leaky),0.799,False
2,Training-only pipeline (correct),0.799,True


## 6. Leakage report and verification

In [6]:
leakage_report = pd.DataFrame([
    ["Scaler fitted before split", "Full-data means and standard deviations", "Put StandardScaler inside Pipeline"],
    ["Imputer fitted before split", "Full-data medians and modes", "Put SimpleImputer inside Pipeline"],
    ["Encoder fitted before split", "Held-out category vocabulary", "Fit OneHotEncoder in Pipeline with handle_unknown='ignore'"],
], columns=["leakage point", "test information exposed", "fix"])
display(leakage_report)

training_imputer = corrected_pipeline.named_steps["preprocessor"].named_transformers_["num"].named_steps["imputer"]
median_check = pd.DataFrame({
    "feature": numeric_columns,
    "training-only statistic": training_imputer.statistics_,
    "full-data median": X[numeric_columns].median().values,
})
display(median_check)

,leakage point,test information exposed,fix
0,Scaler fitted before split,Full-data means and standard deviations,Put StandardScaler inside Pipeline
1,Imputer fitted before split,Full-data medians and modes,Put SimpleImputer inside Pipeline
2,Encoder fitted before split,Held-out category vocabulary,Fit OneHotEncoder in Pipeline with handle_unkn...


,feature,training-only statistic,full-data median
0,Pclass,3.0,3.0000
1,Age,29.0,28.0000
2,SibSp,0.0,0.0000
3,Parch,0.0,0.0000
4,Fare,13.5,14.4542


## 7. Explain and reflect

In [7]:
difference = leaky_accuracy - corrected_accuracy
print(
    f"The fully leaky result differs from the corrected result by {difference:+.3f} accuracy points. "
    "Even a zero or negative difference would not make it valid: the held-out distribution has already "
    "influenced training, so the score is not a clean estimate of generalization."
)
print(
    "Rule for future datasets: split before every learned or data-dependent operation. Fit preprocessing, "
    "feature selection, resampling, and the estimator only on training data—preferably in one Pipeline—and "
    "reserve the test set for final evaluation."
)

The fully leaky result differs from the corrected result by +0.000 accuracy points. Even a zero or negative difference would not make it valid: the held-out distribution has already influenced training, so the score is not a clean estimate of generalization.
Rule for future datasets: split before every learned or data-dependent operation. Fit preprocessing, feature selection, resampling, and the estimator only on training data—preferably in one Pipeline—and reserve the test set for final evaluation.


## Takeaways

- Leakage is a workflow error, whether or not it increases accuracy in one particular split.
- Scaling, imputation, encoding, feature selection, and resampling belong behind the training boundary.
- A pipeline makes that boundary reproducible and protects later cross-validation.
- The held-out test set should remain untouched until final evaluation.